# Runbook: запуск парсеров ClinicalTrials.gov и проверка данных

Эта тетрадь служит единым чек-листом для инженеров данных. В ней собрано всё,
что необходимо для запуска парсеров ClinicalTrials.gov, прогонки тестов и быстрой
визуальной проверки собранных данных перед выгрузкой в дашборд.

## 1. Подготовка окружения

Работа ведётся из корня репозитория `coop_theory_diploma`. Перед первым запуском
рекомендуется создать виртуальное окружение и установить зависимости:

```bash
python -m venv .venv
source .venv/bin/activate
pip install -r requirements.txt
```

In [1]:
!python -m venv .venv
!source .venv/bin/activate
!pip install -r requirements.txt


In [ ]:
# Создание виртуального окружения и установка зависимостей (опционально)
# Уберите символы комментария, чтобы выполнить команды. Активацию окружения
# выполняйте в терминале перед запуском тетради.
# !python -m venv .venv
# !source .venv/bin/activate  # выполните в терминале, чтобы активировать окружение
# !pip install -r requirements.txt


In [ ]:

# Проверим активную версию Python и расположение интерпретатора
import sys
print(sys.version)
print(sys.executable)


> 💡 Если зависимости уже установлены, повторный вызов `pip install` не требуется.
> Ячейка выше оставлена закомментированной для удобства.

## 2. Запуск автотестов для парсеров

В репозитории присутствуют тесты для нормализаторов и клиентских обёрток. Перед
сбором свежих данных стоит убедиться, что они проходят успешно.

In [2]:
!pytest -q

............                                                             [100%]
12 passed in 4.05s


## 3. Запуск высокоуровневых парсеров

Далее можно перейти к сбору данных из ClinicalTrials.gov. Мы ограничиваем число
записей в демонстрационных примерах, чтобы минимизировать нагрузку на API.
При необходимости увеличьте `max_studies`.

In [3]:

from data.parsers.clinical_trials import (
    ClinicalTrialsBySponsorParser,
    ClinicalTrialsExpressionParser,
    DEFAULT_SPONSORS,
)
from data.parsers.clinical_trials_api import ClinicalTrialsClient
import pandas as pd

client = ClinicalTrialsClient()

by_sponsor_parser = ClinicalTrialsBySponsorParser(
    sponsors=list(DEFAULT_SPONSORS)[:3],  # для ускорения берём первые три компании
    client=client,
    max_studies=50,
    batch_size=50,
)

by_sponsor_result = by_sponsor_parser.parse()
by_sponsor_records = by_sponsor_result.payload["records"]

pd.DataFrame(by_sponsor_records)


ClinicalTrialsAPIError: ClinicalTrials.gov returned HTTP 404: <html>
<head><title>404 Not Found</title></head>
<body>
<center><h1>404 Not Found</h1></center>
<hr><center>nginx/1.26.2</center>
</body>
</html>


Помимо агрегирования по компаниям, можно запрашивать произвольные выражения,
например для валидации отдельных направлений исследований.

In [4]:

expression_parser = ClinicalTrialsExpressionParser(
    expr="oncology AND Recruiting",
    client=client,
    max_studies=50,
    batch_size=50,
)

expression_record = expression_parser.parse().payload["records"][0]
expression_record


ClinicalTrialsAPIError: ClinicalTrials.gov returned HTTP 404: <html>
<head><title>404 Not Found</title></head>
<body>
<center><h1>404 Not Found</h1></center>
<hr><center>nginx/1.26.2</center>
</body>
</html>


## 4. Быстрая визуальная проверка агрегатов

После получения данных удобно посмотреть на распределения фаз и статусов, чтобы
убедиться, что парсеры вернули ожидаемые значения.

In [ ]:

phase_counts = (
    pd.DataFrame(by_sponsor_records)
    .set_index("name")["phase_counts"]
    .apply(pd.Series)
    .fillna(0)
    .astype(int)
    .sort_index(axis=1)
)

status_counts = (
    pd.DataFrame(by_sponsor_records)
    .set_index("name")["status_counts"]
    .apply(pd.Series)
    .fillna(0)
    .astype(int)
    .sort_index(axis=1)
)

phase_counts


In [ ]:

import plotly.express as px

phase_long = phase_counts.reset_index().melt(
    id_vars="name", value_name="count", var_name="phase"
)
fig_phase = px.bar(
    phase_long,
    x="name",
    y="count",
    color="phase",
    title="Распределение исследований по фазам",
)
fig_phase.show()

status_long = status_counts.reset_index().melt(
    id_vars="name", value_name="count", var_name="status"
)
fig_status = px.bar(
    status_long,
    x="name",
    y="count",
    color="status",
    title="Распределение исследований по статусам",
)
fig_status.show()


## 5. Сохранение результата в JSON

Для воспроизводимой выгрузки рекомендуется использовать готовый CLI-скрипт.
Пример ниже сохраняет небольшой набор данных в `data/clinical_trials_sample.json`.

In [ ]:

!python data_fetch/build_clinical_trials_dataset.py \
    --sponsor Pfizer \
    --sponsor BIOCAD \
    --max-studies 100 \
    --batch-size 50 \
    --out data/clinical_trials_sample.json


In [ ]:

from pathlib import Path
import json

output_path = Path("data/clinical_trials_sample.json")
with output_path.open("r", encoding="utf-8") as fh:
    dataset = json.load(fh)

dataset.keys()


In [ ]:

pd.DataFrame(dataset.get("sponsors", []))


## 6. Следующие шаги

* При необходимости расширьте список компаний или задайте собственное выражение
  для `ClinicalTrialsExpressionParser`.
* Результаты можно напрямую использовать в дашборде (`app/data.py`) либо
  дополнительно обработать в аналитических тетрадях.
* Не забывайте обновлять зависимости и периодически пересматривать тесты при
  доработках нормализации.